# County Health Forecasting Walkthrough

Forecasting helps county teams move from reporting the past to planning the next few weeks or months. For malaria and similar routine indicators, even a short forward look can support commodity ordering, staffing decisions, outreach planning, and supervision where reporting trends are changing.

This notebook demonstrates how `khis-toolkit` can clean DHIS2 data, run Prophet and ensemble forecasts, compare methods, and surface unusual periods that may deserve review. The public DHIS2 demo server is enough to test the workflow, while KHIS credentials unlock the same process for real Kenya county reporting.

In [ ]:
import khis
from IPython.display import display

conn = khis.connect()
status, message = conn.ping()
print(message)

indicator_matches = khis.list_indicators(conn, search="malaria")
display(indicator_matches.head())
indicator_id = indicator_matches.iloc[0]["id"]
indicator_name = indicator_matches.iloc[0]["name"]

In [ ]:
org_units = conn.get_org_units(level=3)
if org_units.empty:
    org_units = conn.get_org_units()

selected_org_units = org_units[["id", "name"]].head(3)
display(selected_org_units)

df_raw = conn.get_analytics(
    indicator_ids=indicator_id,
    org_unit_ids=selected_org_units["id"].tolist(),
    periods=[f"{year}{month:02d}" for year, month in [
        (2023, 1), (2023, 2), (2023, 3), (2023, 4), (2023, 5), (2023, 6),
        (2023, 7), (2023, 8), (2023, 9), (2023, 10), (2023, 11), (2023, 12),
        (2024, 1), (2024, 2), (2024, 3), (2024, 4), (2024, 5), (2024, 6),
        (2024, 7), (2024, 8), (2024, 9), (2024, 10), (2024, 11), (2024, 12),
    ]],
)
df_clean = khis.clean(df_raw)
df_clean.head()

In [ ]:
county_name = df_clean["org_unit_name"].dropna().iloc[0]

prophet_result = khis.forecast_indicator_series(
    df_clean,
    county=county_name,
    indicator=indicator_name,
    weeks_ahead=4,
    method="prophet",
)
prophet_chart = khis.plot_forecast(prophet_result, title=f"Prophet forecast for {county_name}")
prophet_chart

In [ ]:
ensemble_result = khis.forecast_indicator_series(
    df_clean,
    county=county_name,
    indicator=indicator_name,
    weeks_ahead=4,
    method="ensemble",
)
display(ensemble_result.tail(8))
ensemble_chart = khis.plot_forecast(ensemble_result, title=f"Ensemble forecast for {county_name}")
ensemble_chart

In [ ]:
all_counties_forecast = khis.forecast_all_counties(
    df_clean,
    indicator=indicator_name,
    periods_ahead=4,
    method="ensemble",
)
display(all_counties_forecast[all_counties_forecast["is_forecast"]].head(12))
all_counties_forecast.to_csv("forecast_output.csv", index=False)
print("Saved forecast_output.csv")

In [ ]:
anomalies = khis.anomaly_detection(
    df_clean,
    county=county_name,
    indicator=indicator_name,
)
display(anomalies[anomalies["anomaly_flag"]])

## How a County Health Officer Uses This

The forecast table can be exported and shared during a county review meeting or commodity planning discussion. The key practical question is not whether the model is perfect, but whether it helps the team notice where demand may rise, where an unexpected drop may point to reporting problems, and which counties or sub-counties deserve earlier follow-up.

In practice, a county team would compare the forecast to current stock levels, staffing availability, and known events such as outreach campaigns or disruptions. Unusual periods flagged as anomalies should trigger record checks or facility follow-up before they are used for operational decisions.